# 🛡️ DeepShield Audio — Notebook 03: Model Training

This notebook demonstrates how to train the models using the `tf.data` pipeline and the trainer script.

> **Note:** For full training on the 10GB ASVspoof dataset, it is highly recommended to run the training script from the command line using a GPU: `python -m src.trainer --model custom_cnn`

In [1]:
import sys
sys.path.insert(0, '..')

import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
import os

from src.models.custom_cnn import build_custom_cnn
from src.models.cnn_lstm import build_cnn_lstm
from src.models.efficientnet import build_efficientnet_b0
from src.trainer import build_callbacks
from src.dataset import build_dataset
from src.data_parser import make_synthetic_dataframe

print(f'TensorFlow Version: {tf.__version__}')
print(f'GPU Available: {len(tf.config.list_physical_devices("GPU")) > 0}')

TensorFlow Version: 2.21.0
GPU Available: False


## 1. Prepare Synthetic Data (for demonstration)
Since we might not have the full dataset available in this environment, we'll generate synthetic data to demonstrate the training loop.

In [2]:
# Generate synthetic DataFrame
train_df = make_synthetic_dataframe(n_bonafide=50, n_spoof=50)
val_df   = make_synthetic_dataframe(n_bonafide=20, n_spoof=20)

print(f'Train samples: {len(train_df)}')
print(f'Val samples: {len(val_df)}')

# Create tf.data.Dataset (this will extract log-mel on the fly!)
# Note: On synthetic data where files don't exist, this will error out.
# We will use dummy tensors for the actual `model.fit` demonstration.
print('Dataset pipelines can be built like this:')
print('train_ds = build_dataset(train_df, batch_size=32, is_training=True)')
print('val_ds = build_dataset(val_df, batch_size=32, is_training=False)')

Train samples: 100
Val samples: 40
Dataset pipelines can be built like this:
train_ds = build_dataset(train_df, batch_size=32, is_training=True)
val_ds = build_dataset(val_df, batch_size=32, is_training=False)


## 2. Model Summaries
Let's look at the architectures we've implemented.

In [3]:
print('--- CUSTOM CNN ---')
cnn_model = build_custom_cnn()
cnn_model.summary()

print('\n\n--- CNN-LSTM ---')
lstm_model = build_cnn_lstm()
lstm_model.summary()

--- CUSTOM CNN ---


Model: "DeepShield_CustomCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ log_mel_input (InputLayer)      │ (None, 128, 251, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_1 (Conv2D)                │ (None, 128, 251, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1 (BatchNormalization)        │ (None, 128, 251, 32)   │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu1 (Activation)              │ (None, 128, 251, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_2 (Conv2D)                │ (None, 128, 251, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1_2 (BatchNormalization)      │ (None, 128, 251, 32)   │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu1_2 (Activation)            │ (None, 128, 251, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 64, 125, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop1 (Dropout)                 │ (None, 64, 125, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2_1 (Conv2D)                │ (None, 64, 125, 64)    │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2 (BatchNormalization)        │ (None, 64, 125, 64)    │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu2 (Activation)              │ (None, 64, 125, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2_2 (Conv2D)                │ (None, 64, 125, 64)    │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2_2 (BatchNormalization)      │ (None, 64, 125, 64)    │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu2_2 (Activation)            │ (None, 64, 125, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 32, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop2 (Dropout)                 │ (None, 32, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3_1 (Conv2D)                │ (None, 32, 62, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn3 (BatchNormalization)        │ (None, 32, 62, 128)    │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu3 (Activation)              │ (None, 32, 62, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3_2 (Conv2D)                │ (None, 32, 62, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn3_2 (BatchNormalization)      │ (None, 32, 62, 128)    │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu3_2 (Activation)            │ (None, 32, 62, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool3 (MaxPooling2D)            │ (None, 16, 31, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop3 (Dropout)                 │ (None, 16, 31, 128)    │             

 Total params: 650,465 (2.48 MB)

 Trainable params: 649,057 (2.48 MB)

...

Model: "DeepShield_CNN_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ log_mel_input (InputLayer)      │ (None, 128, 251, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_conv1 (Conv2D)              │ (None, 128, 251, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_bn1 (BatchNormalization)    │ (None, 128, 251, 32)   │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_relu1 (Activation)          │ (None, 128, 251, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_pool1 (MaxPooling2D)        │ (None, 64, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_drop1 (Dropout)             │ (None, 64, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_conv2 (Conv2D)              │ (None, 64, 62, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_bn2 (BatchNormalization)    │ (None, 64, 62, 64)     │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_relu2 (Activation)          │ (None, 64, 62, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_pool2 (MaxPooling2D)        │ (None, 32, 31, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_drop2 (Dropout)             │ (None, 32, 31, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_conv3_gradcam (Conv2D)      │ (None, 32, 31, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_bn3 (BatchNormalization)    │ (None, 32, 31, 128)    │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_relu3 (Activation)          │ (None, 32, 31, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn_drop3 (Dropout)             │ (None, 32, 31, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ permute (Permute)               │ (None, 31, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_seq (Reshape)           │ (None, 31, 4096)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bi_lstm1 (Bidirectional)        │ (None, 31, 128)        │     2,130,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_drop1 (Dropout)            │ (None, 31, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm2 (LSTM)                    │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_drop2 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc_drop (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ prediction (Dense)              │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,281,857 (8.70 MB)

 Trainable params: 2,281,409 (8.70 MB)

 Non-trainable params: 448 (1.75 KB)

## 3. Dummy Training Loop Demonstration
We create some random tensors to simulate a batch of data and run `model.fit()` for 2 epochs.

In [4]:
import numpy as np
from src.config import N_MELS, T_FRAMES

# Generate random noise simulating Log-Mel spectrograms
X_dummy = np.random.randn(64, N_MELS, T_FRAMES, 1).astype(np.float32)
y_dummy = np.random.randint(0, 2, size=(64, 1)).astype(np.float32)

from pathlib import Path
from src.config import SAVED_MODELS_DIR, LOGS_DIR
callbacks = build_callbacks(
    model_name='demo_cnn',
    checkpoint_dir=SAVED_MODELS_DIR / 'demo_cnn',
    tensorboard_dir=LOGS_DIR,
)

history = cnn_model.fit(
    X_dummy, y_dummy,
    batch_size=16,
    epochs=5,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/5



1/4 ━━━━━━━━━━━━━━━━━━━━ 6s 2s/step - auc: 0.7302 - binary_accuracy: 0.6250 - loss: 0.7108 - precision: 0.6000 - recall: 1.0000


2/4 ━━━━━━━━━━━━━━━━━━━━ 0s 404ms/step - auc: 0.6992 - binary_accuracy: 0.6250 - loss: 0.7397 - precision: 0.5769 - recall: 0.9375


3/4 ━━━━━━━━━━━━━━━━━━━━ 0s 431ms/step - auc: 0.5383 - binary_accuracy: 0.5625 - loss: 0.8108 - precision: 0.5714 - recall: 0.6400


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 316ms/step - auc: 0.5200 - binary_accuracy: 0.5490 - loss: 0.8190 - precision: 0.5517 - recall: 0.6154

...


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 322ms/step - auc: 0.6062 - binary_accuracy: 0.6471 - loss: 0.7834 - precision: 0.7500 - recall: 0.4615


Epoch 5: val_loss improved from 0.76979 to 0.76701, saving model to /Users/yash/Documents/Projects/Audio-Deepfake-Detection/saved_models/demo_cnn/demo_cnn_best.keras



Epoch 5: finished saving model to /Users/yash/Documents/Projects/Audio-Deepfake-Detection/saved_models/demo_cnn/demo_cnn_best.keras



4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 365ms/step - auc: 0.6062 - binary_accuracy: 0.6471 - loss: 0.7834 - precision: 0.7500 - recall: 0.4615 - val_auc: 0.4286 - val_binary_accuracy: 0.5385 - val_loss: 0.7670 - val_precision: 0.5385 - val_recall: 1.0000 - learning_rate: 0.0010


Restoring model weights from the end of the best epoch: 5.


## 4. Plot Learning Curves

In [5]:
def plot_learning_curves(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    fig.patch.set_facecolor('#1E1E2E')
    
    for ax in [ax1, ax2]:
        ax.set_facecolor('#1E1E2E')
        ax.tick_params(colors='white')
        ax.xaxis.label.set_color('white')
        ax.yaxis.label.set_color('white')
        ax.title.set_color('white')
        
    # Loss
    ax1.plot(history.history['loss'], label='Train Loss', color='#4ECDC4')
    ax1.plot(history.history['val_loss'], label='Val Loss', color='#E63946')
    ax1.set_title('Model Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend(facecolor='#2A2A35', labelcolor='white')
    
    # Accuracy
    ax2.plot(history.history['binary_accuracy'], label='Train Accuracy', color='#4ECDC4')
    ax2.plot(history.history['val_binary_accuracy'], label='Val Accuracy', color='#E63946')
    ax2.set_title('Model Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Binary Accuracy')
    ax2.legend(facecolor='#2A2A35', labelcolor='white')
    
    plt.show()

plot_learning_curves(history)

/var/folders/gd/mv12hnv97jb8xj3__bq33w0m0000gn/T/ipykernel_45028/4093329350.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
